In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
from tensorflow.keras import regularizers,constraints
from tensorflow.keras.layers import Input, Conv1D, Dense, GlobalAveragePooling1D, BatchNormalization,MaxPooling1D,Dropout,Concatenate,Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import auc, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,StratifiedKFold,KFold
import time
import csv
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei']  
plt.rcParams['axes.unicode_minus'] = False    
import scipy.io as sio
from sklearn.decomposition import PCA
import sys
from contextlib import redirect_stdout

In [ ]:
# ====================  Reading data ========================

class CSVSignalLoader:
    def __init__(self,data_dir,known_classes = 4,unknown_classes = 2,seq_length = 500000):

        self.data_dir =  data_dir 
        self.known_classes = known_classes
        self.unknown_classes = unknown_classes
        self.total_classes = known_classes + unknown_classes
        self.seq_length = seq_length
        
        self.all_data,self.all_labels = self._preload_all_data()

    def _load_csv_to_array(self,filepath):

        arr = []
        with open(filepath,'r') as f:
            for idx,line in enumerate(f):
                if idx>=180:
                    break;
                fields = line.strip().split(',')
                row = [float(x) for x in fields]
                arr.append(row)
        return np.array(arr,dtype=np.float32)
        
    def _process_single_class(self,class_id):
        channel1_path = os.path.join(self.data_dir,f"data_target_{class_id}_I.csv")
        channel2_path = os.path.join(self.data_dir,f"data_target_{class_id}_Q.csv")

        channel1 = self._load_csv_to_array(channel1_path)
        channel2 = self._load_csv_to_array(channel2_path)
        
        merged = np.stack([channel1,channel2],axis=-1)
        print(f"Class {class_id} shape: {merged.shape}")
        return merged
        
    def _preload_all_data(self):
        all_data = {}
        all_labels = {}
        for class_id in range(self.total_classes):
            print(f"Loading class {class_id} data ... ")
            data = self._process_single_class(class_id)
            all_data[class_id]=data
            all_labels[class_id]=np.full(len(data),class_id,dtype = np.int32)
        return all_data,all_labels
    
    def to_onehot(self, y):
        
        n_class = self.known_classes + 1
        y_oh = np.zeros((len(y), n_class), dtype=np.float32)
        for idx, v in enumerate(y):
            if v < self.known_classes:
                y_oh[idx, v] = 1.0
            else:
                y_oh[idx, -1] = 1.0
        return y_oh
    
    def get_kfold_split(self, k=10, train_unknown_ratio=0.04, val_unknown_ratio=0.04,test_known_ratio=0.3, test_unknown_ratio=0.33,random_state=42, batch_size=64):
        """Classify the training set, test set and validation set, and return the final training set"""
        rng = np.random.RandomState(random_state)

        # ======= 1. Separate all samples of known classes and unknown classes ======
        x_known_list = []
        y_known_list = []
        for class_id in range(self.known_classes):
            x_known_list.append(self.all_data[class_id])
            y_known_list.append(self.all_labels[class_id])
        x_known = np.concatenate(x_known_list, axis=0)
        y_known = np.concatenate(y_known_list, axis=0)

        x_unknown_list = []
        y_unknown_list = []
        for class_id in range(self.known_classes, self.total_classes):
            x_unknown_list.append(self.all_data[class_id])
            y_unknown_list.append(self.all_labels[class_id])
        x_unknown = np.concatenate(x_unknown_list, axis=0)
        y_unknown = np.concatenate(y_unknown_list, axis=0)

        n_known_total = len(x_known)
        n_unknown_total = len(x_unknown)
        print(f"Total known samples: {n_known_total}, Total unknown samples: {n_unknown_total}")

        # ========= 2. Partition independent test set (completely separate from cross-validation) ==========
        n_test_known = int(n_known_total * test_known_ratio)
        test_known_idx = rng.choice(n_known_total, n_test_known, replace=False)
        x_test_known = x_known[test_known_idx]
        y_test_known = y_known[test_known_idx]

        trainval_known_mask = np.ones(n_known_total, dtype=bool)
        trainval_known_mask[test_known_idx] = False
        x_trainval_known = x_known[trainval_known_mask]
        y_trainval_known = y_known[trainval_known_mask]

        n_test_total = int(n_test_known / (1 - test_unknown_ratio))
        n_test_unknown = min(n_test_total - n_test_known, n_unknown_total)
        test_unknown_idx = rng.choice(n_unknown_total, n_test_unknown, replace=False)
        x_test_unknown = x_unknown[test_unknown_idx]
        y_test_unknown = y_unknown[test_unknown_idx]

        trainval_unknown_mask = np.ones(n_unknown_total, dtype=bool)
        trainval_unknown_mask[test_unknown_idx] = False
        x_trainval_unknown = x_unknown[trainval_unknown_mask]
        y_trainval_unknown = y_unknown[trainval_unknown_mask]

        
        x_test = np.concatenate([x_test_known, x_test_unknown], axis=0)
        y_test = np.concatenate([y_test_known, y_test_unknown], axis=0)
        y_test_oh = self.to_onehot(y_test)
        test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test_oh)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

        print(f"\n=== Independent Test Set ===")
        print(f"Total test samples: {len(x_test)}")
        print(f"  Known: {len(x_test_known)}, Unknown: {len(x_test_unknown)}")
        print(f"  Unknown ratio: {len(x_test_unknown)/len(x_test):.4f}\n")

        # ========= 3. Cross-validation partitioning (global no duplicate assignment of unknown classes) ==========
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)
        split_indices = list(skf.split(x_trainval_known, y_trainval_known))

        folds_unknown_needs = []
        total_unknown_needed = 0
        for train_idx, val_idx in split_indices:
            n_train_known = len(train_idx)
            n_val_known = len(val_idx)
            n_train_unknown = int(n_train_known / (1 - train_unknown_ratio)) - n_train_known
            n_train_unknown = max(0, min(n_train_unknown, len(x_trainval_unknown)))
            n_val_unknown = int(n_val_known / (1 - val_unknown_ratio)) - n_val_known
            n_val_unknown = max(0, min(n_val_unknown, len(x_trainval_unknown)))
            folds_unknown_needs.append((n_train_unknown, n_val_unknown))
            total_unknown_needed += n_train_unknown + n_val_unknown

        if total_unknown_needed > len(x_trainval_unknown):
            raise ValueError(f"Total unknown samples needed for CV ({total_unknown_needed}) exceeds pool size ({len(x_trainval_unknown)}).")

        # Disrupt the unknown class pool and prepare for global non-duplicate allocation
        shuffle_idx = rng.permutation(len(x_trainval_unknown))
        x_unknown_shuffled = x_trainval_unknown[shuffle_idx]
        y_unknown_shuffled = y_trainval_unknown[shuffle_idx]

        ptr = 0
        folds = []
        total_cv_known = len(x_trainval_known)
        total_cv_unknown_pool = len(x_trainval_unknown)

        for fold_idx, ((train_idx, val_idx), (n_train_unknown, n_val_unknown)) in enumerate(zip(split_indices, folds_unknown_needs)):
            x_train_known = x_trainval_known[train_idx]
            y_train_known = y_trainval_known[train_idx]
            x_val_known = x_trainval_known[val_idx]
            y_val_known = y_trainval_known[val_idx]

            train_unknown_end = ptr + n_train_unknown
            x_train_unknown = x_unknown_shuffled[ptr:train_unknown_end]
            y_train_unknown = y_unknown_shuffled[ptr:train_unknown_end]
            ptr = train_unknown_end

            val_unknown_end = ptr + n_val_unknown
            x_val_unknown = x_unknown_shuffled[ptr:val_unknown_end]
            y_val_unknown = y_unknown_shuffled[ptr:val_unknown_end]
            ptr = val_unknown_end

            x_train = np.concatenate([x_train_known, x_train_unknown], axis=0)
            y_train = np.concatenate([y_train_known, y_train_unknown], axis=0)
            x_val = np.concatenate([x_val_known, x_val_unknown], axis=0)
            y_val = np.concatenate([y_val_known, y_val_unknown], axis=0)

            y_train_oh = self.to_onehot(y_train)
            y_val_oh = self.to_onehot(y_val)

            train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train_oh)).shuffle(500).batch(batch_size).prefetch(tf.data.AUTOTUNE)
            val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val_oh)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

            folds.append((train_ds, val_ds))

            print(f"Fold {fold_idx+1}:")
            print(f"  Train: total={len(x_train)}, known={n_train_known}, unknown={n_train_unknown}, "
                  f"unknown_ratio={n_train_unknown/len(x_train):.4f}")
            print(f"  Val:   total={len(x_val)}, known={n_val_known}, unknown={n_val_unknown}, "
                  f"unknown_ratio={n_val_unknown/len(x_val):.4f}")

        print(f"\n=== Cross-Validation Data Pool (excludes independent test set) ===")
        print(f"Total known samples available for CV: {total_cv_known}")
        print(f"Total unknown samples available for CV (pool size): {total_cv_unknown_pool}")
        print(f"Total unknown samples used across all folds: {ptr}")
        print(f"Note: Unknown samples are globally shuffled and allocated without replacement across folds and between train/val sets.\n")

        # ========= 4. Construct the final training set  ==========
        n_train_final_known = len(x_trainval_known)
        n_train_final_total_target = int(n_train_final_known / (1 - train_unknown_ratio))
        n_train_final_unknown = n_train_final_total_target - n_train_final_known
        n_train_final_unknown = min(n_train_final_unknown, len(x_trainval_unknown))

        # Randomly sample from unknown class pool (without replacement)
        final_unknown_idx = rng.choice(len(x_trainval_unknown), n_train_final_unknown, replace=False)
        x_train_final_unknown = x_trainval_unknown[final_unknown_idx]
        y_train_final_unknown = y_trainval_unknown[final_unknown_idx]

        x_train_final = np.concatenate([x_trainval_known, x_train_final_unknown], axis=0)
        y_train_final = np.concatenate([y_trainval_known, y_train_final_unknown], axis=0)
        y_train_final_oh = self.to_onehot(y_train_final)

        # Statistics of final training set information
        total_final = len(x_train_final)
        unknown_final_count = len(x_train_final_unknown)
        unknown_final_ratio = unknown_final_count / total_final
        print(f"=== Final Training Set (for final model training) ===")
        print(f"Total samples: {total_final}")
        print(f"  Known: {n_train_final_known}, Unknown: {unknown_final_count}")
        print(f"  Unknown ratio: {unknown_final_ratio:.4f} (target: {train_unknown_ratio:.2%})")
        print(f"Note: All known samples from CV pool are used; unknown samples are sampled without replacement.\n")

        
        final_train_ds = tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final_oh)).shuffle(2000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

        
        return folds, test_ds, final_train_ds





          

In [ ]:
#  =================   Establish the network ==================

class ObjectosphereRecognizer(tf.keras.Model):
    def __init__(self, num_known_classes=4, seq_length=500000, channels=2, margin=0.6, alpha=0.1, beta=0.1, ortho_weight = 0.0):
        super().__init__()
        self.num_known_classes = num_known_classes
        self.seq_length = seq_length
        self.channels = channels
        self.margin = margin  
        self.alpha = alpha    
        self.beta = beta      
        self.ortho_weight = ortho_weight   
        
        self.feature_dim = 32  
        self.class_prototypes = None  
        
        
        inputs = Input(shape=(seq_length, channels))  # [batch,500000,2]
        
        I = inputs[:,:,0:1]  #[batch,500000,1]
        Q = inputs[:,:,1:2] 
        
        conv_I_linear = Conv1D(32,16,strides=8,padding='same',activation='relu')(I)
        conv_Q_linear = Conv1D(32,16,strides=8,padding='same',activation='relu')(Q)
        

        
        
        def cos_sim_loss(inputs):
            conv_I_linear,conv_Q_linear = inputs
            batch_size = tf.shape(conv_I_linear)[0]
            I_flat = tf.reshape(conv_I_linear,[batch_size,-1])
            Q_flat = tf.reshape(conv_Q_linear,[batch_size,-1])
            I_norm = tf.nn.l2_normalize(I_flat,axis=1)
            Q_norm = tf.nn.l2_normalize(Q_flat,axis=1)
            cos_sim = tf.reduce_mean(tf.maximum(0.0,tf.reduce_sum(I_norm*Q_norm,axis=1)))
            return cos_sim
        ortho_output = Lambda(cos_sim_loss,name='orthogonality_output')([conv_I_linear,conv_Q_linear])

        
        conv_I_linear = MaxPooling1D(pool_size=8)(conv_I_linear)
        conv_Q_linear = MaxPooling1D(pool_size=8)(conv_Q_linear)
        
        global_I = GlobalAveragePooling1D()(conv_I_linear)
        global_Q = GlobalAveragePooling1D()(conv_Q_linear)

        
        merged = Concatenate()([global_I,global_Q])

    
        x = Dense(64,activation='relu')(merged)
       
        features = Dense(self.feature_dim,activation='relu')(x)
        
        features = Lambda(lambda t: tf.nn.l2_normalize(t, axis=1))(features)
    
        logits = Dense(num_known_classes)(features)
        probs = Lambda(lambda t: tf.nn.softmax(t, axis=1))(logits)

        
    
        self._inference_func = Model(inputs=inputs, outputs=[probs,features,ortho_output])
    
    def call(self, inputs, training=False):
        probs, features, ortho_output = self._inference_func(inputs, training=training)
        return probs, features, ortho_output
    

    def train_step(self, data):
        x, y = data
    
        known_mask = tf.reduce_max(y[:, :self.num_known_classes], axis=1) > 0
        unknown_mask = y[:, self.num_known_classes] > 0
    
        known_count = tf.reduce_sum(tf.cast(known_mask, tf.float32))
        unknown_count = tf.reduce_sum(tf.cast(unknown_mask, tf.float32))
        total_count = known_count + unknown_count
    
        known_weight = tf.math.divide_no_nan(total_count, 2 * known_count)
        unknown_weight = tf.math.divide_no_nan(total_count, 2 * unknown_count)
    
        with tf.GradientTape() as tape:
            probs, features, ortho_output = self(x, training=True)
            
            if self.class_prototypes is None:
                self.class_prototypes = tf.Variable(
                    tf.zeros([self.num_known_classes, self.feature_dim]),
                    trainable=False, dtype=tf.float32
                )
        
            label_indices = tf.argmax(y[:, :self.num_known_classes], axis=1)
            
            masked_ce_loss = tf.keras.losses.categorical_crossentropy(
                y[:, :self.num_known_classes], probs) * tf.cast(known_mask, tf.float32)
            ce_loss = tf.math.divide_no_nan(tf.reduce_sum(masked_ce_loss), known_count) * known_weight
        
            entropy = -tf.reduce_sum(probs * tf.math.log(probs + 1e-10), axis=1)
            masked_entropy = entropy * tf.cast(unknown_mask, tf.float32)
            entropy_loss = -tf.math.divide_no_nan(tf.reduce_sum(masked_entropy), unknown_count) * unknown_weight

            for c in range(self.num_known_classes):
                class_mask = tf.logical_and(tf.equal(label_indices,c), known_mask)
                class_count = tf.reduce_sum(tf.cast(class_mask, tf.float32))
            
                def update_prototype():
                    class_features = features * tf.expand_dims(tf.cast(class_mask,tf.float32),axis = 1)
                    class_center = tf.reduce_sum(class_features,axis = 0)/class_count
                    class_center = tf.nn.l2_normalize(class_center, axis=0)
                
                    momentum = 0.1
                    updated_prototype = (1-momentum) * self.class_prototypes[c] + momentum * class_center
                    return tf.nn.l2_normalize(updated_prototype, axis=0)
            
                def no_update():
                    return self.class_prototypes[c]
            
                
                updated = tf.cond(class_count > 0, update_prototype, no_update)
                self.class_prototypes[c].assign(updated)
        
            batch_prototypes = tf.gather(self.class_prototypes, label_indices)
            cosine_similarity = tf.reduce_sum(features * batch_prototypes, axis=1)
        
            known_feature_loss = tf.reduce_sum((1.0-cosine_similarity)*tf.cast(known_mask,tf.float32))/tf.maximum(known_count,1.0)
        
            sim_matrix = tf.matmul(features, tf.transpose(self.class_prototypes))
            unknown_max_sim = tf.reduce_max(sim_matrix, axis=1)
        
            unknown_feature_loss = tf.math.divide_no_nan(tf.reduce_sum(unknown_max_sim*tf.cast(unknown_mask,tf.float32)),unknown_count)
        
            feature_loss = known_feature_loss + 0.1 * unknown_feature_loss
        
            total_loss = ce_loss + self.alpha * entropy_loss + self.beta * feature_loss + self.ortho_weight * ortho_output
            
        

        trainable_vars = self.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
    
        known_sim = tf.math.divide_no_nan(
            tf.reduce_sum(cosine_similarity * tf.cast(known_mask, tf.float32)), 
            known_count
        )
        
        unknown_max_sim = tf.reduce_max(tf.matmul(
            features * tf.expand_dims(tf.cast(unknown_mask,tf.float32),axis=1),
            tf.transpose(self.class_prototypes)
        ),axis = 1)
        
        unknown_sim = tf.math.divide_no_nan(
            tf.reduce_sum(unknown_max_sim),
            unknown_count
        )
    
        self.compiled_metrics.update_state(y[:, :self.num_known_classes], probs)
        metrics = {m.name: m.result() for m in self.metrics}
        metrics.update({
            'loss': total_loss,
            'ce_loss': ce_loss,
            'entropy_loss': entropy_loss,
            'feature_loss': feature_loss,
            'ortho_loss':ortho_output,
            'known_sim': known_sim,
            'unknown_sim': unknown_sim
        })
    
        return metrics

    def test_step(self, data):
        x, y = data
        probs, features, ortho_output = self(x, training=False)
        
        
        loss = tf.keras.losses.categorical_crossentropy(y[:, :self.num_known_classes], probs)
        loss = tf.reduce_mean(loss)

        self.compiled_metrics.update_state(y[:, :self.num_known_classes], probs)

        known_mask = tf.reduce_max(y[:, :self.num_known_classes], axis=1) > 0
        unknown_mask = y[:, self.num_known_classes] > 0

        known_count = tf.reduce_sum(tf.cast(known_mask, tf.float32))
        unknown_count = tf.reduce_sum(tf.cast(unknown_mask, tf.float32))
        if hasattr(self,'class_prototypes') and self.class_prototypes is not None:
            sim_matrix = tf.matmul(features,tf.transpose(self.class_prototypes))
            
            label_indices = tf.argmax(y[:,:self.num_known_classes],axis = 1)
            label_indices = tf.cast(label_indices,tf.int32)
            
            val_known_sim_mean = tf.gather_nd(sim_matrix,tf.stack([tf.range(tf.shape(label_indices)[0]),label_indices],axis = 1))
                        
            unknown_max_sim = tf.reduce_max(sim_matrix,axis = 1)
        
        
            metrics = {m.name: m.result() for m in self.metrics}
            metrics.update({
                'loss':loss,
                'val_known_sim_mean':tf.math.divide_no_nan(tf.reduce_sum(val_known_sim_mean * tf.cast(known_mask,tf.float32)),known_count),
                'val_unknown_sim_mean':tf.math.divide_no_nan(tf.reduce_sum(unknown_max_sim * tf.cast(unknown_mask,tf.float32)),unknown_count)
            })

        return metrics


    def predict_with_features(self, x, batch_size=64):
        probs, features,ortho_output = self._inference_func.predict(x, verbose=0, batch_size=batch_size)
        return probs, features,ortho_output

    def predict_with_rejection(self, x, threshold=0.4, use_similarity=True, batch_size=64):

        probs, features, ortho_output = self.predict_with_features(x, batch_size=batch_size)
        pred_label = np.argmax(probs, axis=1)
        pred_max = np.max(probs, axis=1)
        
        if use_similarity and hasattr(self, 'class_prototypes') and self.class_prototypes is not None:
            class_prototypes_np = self.class_prototypes.numpy()
            sim_matrix = np.dot(features, class_prototypes_np.T)
            
            pred_similarities = np.array([sim_matrix[i, pred_label[i]] for i in range(len(pred_label))])
            
            confidence_scores = pred_max * pred_similarities
            pred_label[confidence_scores < threshold] = -1
            return pred_label, confidence_scores
        else:
            pred_label[pred_max < threshold] = -1
            return pred_label, pred_max
            
    def evaluate_oscr(self, test_ds, save_path,threshold=0.4, use_similarity=True):
        
        if use_similarity:
            log_path = os.path.join(save_path,"oscr_log_use_similarity.txt")
        else:
            log_path = os.path.join(save_path,"oscr_log_no_similarity.txt")
        
        with open(log_path,"w") as f, redirect_stdout(f):
            print("=== OSCR Evaluate Info ===")
            
            
            all_probs = []
            all_features = []
            all_true_labels = []
            
            total_samples = 0
            for x_batch,y_batch in test_ds:
                batch_size = x_batch.shape[0]
                total_samples += batch_size
                
                if isinstance(x_batch,tf.Tensor):
                    x_batch = x_batch.numpy()
                
                probs_batch, features_batch, _ = self.predict_with_features(x_batch)
                # one-hot 
                if isinstance(y_batch,tf.Tensor):
                    y_batch = y_batch.numpy()
                y_int = np.argmax(y_batch,axis = 1)
                
                all_probs.append(probs_batch)
                all_features.append(features_batch)
                all_true_labels.append(y_int)
            
        
            probs = np.concatenate(all_probs,axis = 0)
            features = np.concatenate(all_features,axis = 0)
            true_labels = np.concatenate(all_true_labels,axis = 0)
            print(f"total test samples: {total_samples}")
            print(f"Predicted probability shape : {probs.shape}, features shape: {features.shape}")
            
            pred_labels = np.argmax(probs,axis = 1)
            pred_max = np.max(probs,axis = 1)
            
            
            if use_similarity and hasattr(self, 'class_prototypes') and self.class_prototypes is not None:
            
                class_prototypes_np = self.class_prototypes.numpy()
                sim_matrix = np.dot(features, class_prototypes_np.T)
            
                pred_similarities = np.array([sim_matrix[i, pred_labels[i]] for i in range(len(pred_labels))])
            
                confidence_scores = pred_max * pred_similarities
                print("Use feature similarity + probability as confidence")
            else:
        
                confidence_scores = pred_max
                print("Use only probabilities as confidence levels")
            
            all_thresholds = np.linspace(0, np.max(confidence_scores), 100)
            ccr, fpr = [], []
            for th in all_thresholds:
                pred_openset = pred_labels.copy()
                pred_openset[confidence_scores < th] = -1
                true_is_known = true_labels < self.num_known_classes
                true_is_unknown = true_labels >= self.num_known_classes
            
                # CCR
                ccr_val = np.mean((pred_openset == true_labels)[true_is_known]) if np.any(true_is_known) else 0.0
            
                # FPR
                fpr_val = np.mean((pred_openset != -1)[true_is_unknown]) if np.any(true_is_unknown) else 0.0
            
                ccr.append(ccr_val)
                fpr.append(fpr_val)
            
            auc_score = auc(fpr, ccr)

            
            th_best = threshold
            pred_openset = pred_labels.copy()
            pred_openset[confidence_scores < th_best] = -1
        
            true_is_known = true_labels < self.num_known_classes
            true_is_unknown = true_labels >= self.num_known_classes
        
            known_acc = np.mean((pred_openset == true_labels)[true_is_known]) if np.any(true_is_known) else 0.0
            unknown_acc = np.mean((pred_openset == -1)[true_is_unknown]) if np.any(true_is_unknown) else 0.0
        
            print(f"Known acc: {known_acc:.4f}, Unknown rejection acc: {unknown_acc:.4f}")
            y_true_eval = true_labels.copy()
            y_true_eval[true_is_unknown] = -1
            print("Confusion matrix (known category labels and unknown(-1)):\n")
            print(confusion_matrix(y_true_eval, pred_openset, labels=list(range(self.num_known_classes))+[-1]))

            print(f"Validation set probability distribution - min: {np.min(pred_max):.4f}, max: {np.max(pred_max):.4f}, mean: {np.mean(pred_max):.4f}")
            print(f"Probability distribution example: {probs[0]}")
        

            if use_similarity and hasattr(self, 'class_prototypes') and self.class_prototypes is not None:
                print(f"Feature similarity distribution-known class: {np.mean(pred_similarities[true_is_known]):.2f}±{np.std(pred_similarities[true_is_known]):.2f}, "
                      f"Unknown class: {np.mean(pred_similarities[true_is_unknown]):.2f}±{np.std(pred_similarities[true_is_unknown]):.2f}")
            print(f"OSCR AUC: {auc_score:.4f}")
            
        return known_acc, unknown_acc, auc_score




In [ ]:
def visualize_feature_sphere(features, labels, class_names=None, unknown_class_index=4, save_path=None):
    
    log_path = None
    if save_path:
        log_path = os.path.splitext(save_path)[0]+"_log.txt"
    
    if log_path:
        log_file = open(log_path,"w")
        print(f"Debugging information will be saved to: {log_path}")
        f_redirect = redirect_stdout(log_file)
    else:
        f_redirect = redirect_stdout(sys.stdout)

    with f_redirect:
        print("=== Feature Sphere Visulization Debug Info ===")
        print(f"Input features shape: {features.shape}")
        print(f"Input labels shape: {labels.shape}")

        if len(labels.shape) > 1 and labels.shape[1] == 5:
            print("Convert 5-digit one-hot encoding to category index")

            unknown_samples = labels[:, unknown_class_index] == 1
        
            class_indices = np.argmax(labels, axis=1)
        
            if np.any(unknown_samples):
                print(f"Detect {np.sum(unknown_samples)} unknown class samples")
                class_indices[unknown_samples] = -1 
        
            labels = class_indices
    
        print(f"Label shape after conversion: {labels.shape}")
        print(f"Unique tag value: {np.unique(labels)}")
    
        features_norm = np.linalg.norm(features, axis=1, keepdims=True)
        features_norm = np.maximum(features_norm, 1e-10)
        features = features / features_norm
    
        if features.shape[1] > 3:
            pca = PCA(n_components=3)
            features_3d = pca.fit_transform(features)
            print(f"feature shape after PCA: {features_3d.shape}")

            explained_variance = pca.explained_variance_ratio_
            print(f"Proportion of variance explained by PCA: {explained_variance}")
            print(f"Cumulative explained variance: {np.sum(explained_variance):.4f}")
        else:
            features_3d = features
            print(f"Use original feature shape: {features_3d.shape}")
    
        
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection='3d')
    
        
        unique_labels = np.unique(labels)
        print(f"unique tag: {unique_labels}")
    
        colors = plt.cm.jet(np.linspace(0, 1, len(unique_labels)))
    
        u = np.linspace(0, 2 * np.pi, 100)
        v = np.linspace(0, np.pi, 100)
        x = 0.99 * np.outer(np.cos(u), np.sin(v))
        y = 0.99 * np.outer(np.sin(u), np.sin(v))
        z = 0.99 * np.outer(np.ones(np.size(u)), np.cos(v))
        ax.plot_surface(x, y, z, color='gray', alpha=0.1)
    
        for i, label in enumerate(unique_labels):
            
            idx = labels == label
            print(f"Label {label} The index array shape of: {idx.shape}, samples num: {np.sum(idx)}")
        
            
            if np.sum(idx) == 0:
                print(f"Warning: label {label} No sample, skip")
                continue
            
            
            current_features = features_3d[idx]
        
        
            print(f"Label {label} feature shape: {current_features.shape}")
        
            if label == -1:
                label_name = "Unknown"
            elif class_names is not None and int(label) < len(class_names):
                label_name = class_names[int(label)]
            else:
                label_name = f"Class {int(label)}"
        
            
            marker = 'x' if label == -1 else 'o'
            size = 50 if label == -1 else 30
            alpha = 0.9 if label == -1 else 0.7
        
            
            ax.scatter(
                current_features[:, 0], 
                current_features[:, 1], 
                current_features[:, 2],
                c=[colors[i]], 
                marker=marker,
                s=size,
                alpha=alpha,
                label=label_name
            )
    
    
        ax.legend(loc='best', fontsize=12)
        ax.set_title('Feature Distribution on Unit Hypersphere (PCA 3D)', fontsize=16)
    
    
        ax.set_xlim([-1.1, 1.1])
        ax.set_ylim([-1.1, 1.1])
        ax.set_zlim([-1.1, 1.1])
        ax.set_xlabel('PC1', fontsize=14)
        ax.set_ylabel('PC2', fontsize=14)
        ax.set_zlabel('PC3', fontsize=14)
    
        
        ax.grid(True)
    
        plt.tight_layout()
    
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
        plt.close()
    
    if log_path:
        log_file.close()


In [ ]:
# loading data

seq_length = 500000  
num_known_classes = 4
batch_size = 32
k_folds = 10
data_loader = CSVSignalLoader(data_dir = "Public_RF_open_set_6", known_classes=num_known_classes,unknown_classes=2,seq_length = seq_length)


In [ ]:
train_unknown_ratio=0.035
val_unknown_ratio=0.2
test_known_ratio=0.2
test_unknown_ratio=0.2
random_state=42

folds,test_ds,final_train_ds = data_loader.get_kfold_split(k_folds,train_unknown_ratio,val_unknown_ratio,test_known_ratio, test_unknown_ratio,random_state,batch_size)


In [ ]:
# save data
def save_history_to_csv(history_data, filename,save_dir):
    with open(os.path.join(save_dir, filename), "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(np.mat(history_data).transpose().tolist())


def save_result_fold(save_dir,history):
    Accuracy = history.history['accuracy']
    Val_Accuracy = history.history['val_accuracy']
    plt.figure(figsize=(10, 6))
    plt.plot(Accuracy, 'b', label='Training Accuracy')
    plt.plot(Val_Accuracy, 'r', label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'Accuracy.png'))
    plt.close()
    
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(loss, 'b', label='Training Loss')
    plt.plot(val_loss, 'r', label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'Loss.png'))
    plt.close()

    known_sim_mean = history.history['known_sim']
    unknown_sim_mean = history.history['unknown_sim']
    plt.figure(figsize=(10, 6))
    plt.plot(known_sim_mean, 'b', label='Known sim mean')
    plt.plot(unknown_sim_mean, 'r', label='Unknown sim mean')
    plt.xlabel('Epochs')
    plt.ylabel('Train set sim mean')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'Train set sim mean.png'))
    plt.close()
    
    val_known_sim_mean = history.history['val_val_known_sim_mean']
    val_unknown_sim_mean = history.history['val_val_unknown_sim_mean']
    plt.figure(figsize=(10, 6))
    plt.plot(val_known_sim_mean, 'b', label='Val Known sim mean')
    plt.plot(val_unknown_sim_mean, 'r', label='Val Unknown sim mean')
    plt.xlabel('Epochs')
    plt.ylabel('Val set sim mean')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'Val set sim mean.png'))
    plt.close()

    ce_loss = history.history['ce_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(ce_loss, 'b', label='ce_loss')
    plt.xlabel('Epochs')
    plt.ylabel('ce_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'ce_loss.png'))
    plt.close()
    
    entropy_loss = history.history['entropy_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(entropy_loss, 'b', label='entropy_loss')
    plt.xlabel('Epochs')
    plt.ylabel('entropy_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'entropy_loss.png'))
    plt.close()
    
    feature_loss = history.history['feature_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(feature_loss, 'b', label='feature_loss')
    plt.xlabel('Epochs')
    plt.ylabel('feature_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'feature_loss.png'))
    plt.close()

    ortho_loss = history.history['ortho_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(ortho_loss, 'b', label='ortho_loss')
    plt.xlabel('Epochs')
    plt.ylabel('ortho_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'ortho_loss.png'))
    plt.close()

    
    save_history_to_csv(history.history['accuracy'], 'Class_Accuracy.csv',save_dir)
    save_history_to_csv(history.history['loss'], 'loss.csv',save_dir)
    save_history_to_csv(history.history['known_sim'], 'Train_known_sim_mean.csv',save_dir)
    save_history_to_csv(history.history['unknown_sim'], 'Train_unknown_sim_mean.csv',save_dir)
    
    save_history_to_csv(history.history['ce_loss'], 'ce_loss.csv',save_dir)
    save_history_to_csv(history.history['entropy_loss'], 'entropy_loss.csv',save_dir)
    save_history_to_csv(history.history['feature_loss'], 'feature_loss.csv',save_dir)
    save_history_to_csv(history.history['ortho_loss'], 'ortho_loss.csv',save_dir)
    
    save_history_to_csv(history.history['val_loss'], 'val_loss.csv',save_dir)
    save_history_to_csv(history.history['val_accuracy'], 'val_accuracy.csv',save_dir)
    
    save_history_to_csv(history.history['val_val_known_sim_mean'], 'val_known_sim_mean.csv',save_dir)
    save_history_to_csv(history.history['val_val_unknown_sim_mean'], 'val_unknown_sim_mean.csv',save_dir)

In [ ]:
known_acc_list1 = []
unknown_acc_list1 = []
oscr_list1 = []
oscr_list2 = []
known_acc_list2 = []
unknown_acc_list2 = []

ortho_weight_set = 0.0
nb_epoch = 1000
margin = 0.6
loss_unknown_ratio = 0.1
t0 = time.time()
loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
base_path = f"Public_RF_Openset_Classify/Margin_{margin}_lossunknownratio{loss_unknown_ratio}_Batchsize_{batch_size}_Nbepoch_{nb_epoch}/{loca}-{ortho_weight_set}"

for fold_idx,(train_ds,val_ds) in enumerate(folds):
    print(f"\n=== Fold {fold_idx+1}/{k_folds} ===")
    fold_loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
    fold_path = f"{fold_loca}-fold{fold_idx}"
    fold_dir = os.path.join(base_path,fold_path)
    os.makedirs(fold_dir,exist_ok=True)
    
    tf.keras.backend.clear_session()

    model = ObjectosphereRecognizer(
        num_known_classes = num_known_classes,
        seq_length = seq_length,
        channels = 2,
        margin = 0.6,
        alpha = 0.1,
        beta = 0.05,
        ortho_weight = ortho_weight_set
    )

    model.compile(optimizer = tf.keras.optimizers.Adam(0.001),metrics=['accuracy'])

    history = model.fit(
        train_ds,
        validation_data = val_ds,
        epochs = nb_epoch,
        verbose = 0)
    
    save_result_fold(fold_dir,history)
    
    features_list = []
    probs_list = []
    orth_loss_list = []
    labels_list = []
    
    for x_batch,y_batch in val_ds:
        probs_sub, features_sub, orth_loss_sub = model(x_batch,training = False)
        features_list.append(features_sub.numpy())
        probs_list.append(probs_sub.numpy())
        orth_loss_list.append(orth_loss_sub.numpy())
        labels_list.append(y_batch.numpy())
    
    
    features_val = np.concatenate(features_list,axis = 0)
    probs_val = np.concatenate(probs_list,axis = 0)
    orth_loss_val = np.array(orth_loss_list)
    y_val_fold_oh = np.concatenate(labels_list,axis = 0)
    

    class_names = [f'Class {i}' for i in range(4)]
    class_names.append('Unknown')
    mat_file_path = os.path.join(fold_dir,'features_and_labels.mat')
    sio.savemat(mat_file_path,{
        'features':features_val,
        'labels':y_val_fold_oh,
        'class_names':class_names
    })
    
    visualize_feature_sphere(
        features_val,
        y_val_fold_oh,
        class_names=class_names,
        unknown_class_index=num_known_classes,
        save_path=f'{fold_dir}/feature_sphere_visualization.png'
    )

    known_acc1,unknown_acc1,auc_score = model.evaluate_oscr(val_ds,fold_dir,threshold=0.4,use_similarity=True)
    known_acc_list1.append(known_acc1)
    unknown_acc_list1.append(unknown_acc1)
    oscr_list1.append(auc_score)
    
    known_acc2,unknown_acc2,auc_score2 = model.evaluate_oscr(val_ds,fold_dir,threshold=0.4,use_similarity = False)
    known_acc_list2.append(known_acc2)
    unknown_acc_list2.append(unknown_acc2)
    oscr_list2.append(auc_score2)

    summary_msg = (
        f"==== Fold {fold_idx+1} OSCR AUC ====\n"
        f"Feature magnitude + probability: {auc_score:.4f}\n"
        f"Only probability: {auc_score2:.4f}"
    )
    summary_path = os.path.join(fold_dir,"fold_val_OSCR_summart.txt")
    with open(summary_path,"w",encoding = "utf-8") as f:
        f.write(summary_msg)
    

t1=time.time()
elapsed_time = t1-t0
print(f"Time: {elapsed_time:.4f} s")



In [ ]:

print("\n==== ALL-fold Results ====")
known_acc1_mean, known_acc1_std = np.mean(known_acc_list1), np.std(known_acc_list1)
known_acc2_mean, known_acc2_std = np.mean(known_acc_list2), np.std(known_acc_list2)

unknown_acc1_mean, unknown_acc1_std = np.mean(unknown_acc_list1), np.std(unknown_acc_list1)
unknown_acc2_mean, unknown_acc2_std = np.mean(unknown_acc_list2), np.std(unknown_acc_list2)

oscr_1_mean, oscr_1_std = np.mean(oscr_list1), np.std(oscr_list1)
oscr_2_mean, oscr_2_std = np.mean(oscr_list2), np.std(oscr_list2)

print(f"Feature magnitude + probability Known_acc: {known_acc1_mean:.4f} ± {known_acc1_std:.4f}")
print(f"Feature magnitude + probability UnKnown_acc: {unknown_acc1_mean:.4f} ± {unknown_acc1_std:.4f}")
print(f"Only probability Known_acc: {known_acc2_mean:.4f} ± {known_acc2_std:.4f}")
print(f"Only probability UnKnown_acc: {unknown_acc2_mean:.4f} ± {unknown_acc2_std:.4f}")
print(f"Feature magnitude + probability OSCR: {oscr_1_mean:.4f} ± {oscr_1_std:.4f}")
print(f"Only probability OSCR: {oscr_2_mean:.4f} ± {oscr_2_std:.4f}")

summary_msg = (
    "==== ALL-fold Results ====\n"
    f"Feature magnitude + probability Known_acc: {known_acc1_mean:.4f} ± {known_acc1_std:.4f}\n"
    f"Feature magnitude + probability UnKnown_acc: {unknown_acc1_mean:.4f} ± {unknown_acc1_std:.4f}\n"
    f"Feature magnitude + probability OSCR: {oscr_1_mean:.4f} ± {oscr_1_std:.4f}\n"
    f"Only probability Known_acc: {known_acc2_mean:.4f} ± {known_acc2_std:.4f}\n"
    f"Only probability OSCR: {oscr_2_mean:.4f} ± {oscr_2_std:.4f}\n"
    f"Only probability UnKnown_acc: {unknown_acc2_mean:.4f} ± {unknown_acc2_std:.4f}"
)
summary_path = os.path.join(base_path, "results_summary.txt")


with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg)

print(f"Save to: {summary_path}")


In [ ]:


def save_result_test(save_dir,history):
    Accuracy = history.history['accuracy']
    plt.figure(figsize=(10, 6))
    plt.plot(Accuracy, 'b', label='Training Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'Accuracy.png'))
    plt.close()

    known_sim = history.history['known_sim']
    unknown_sim = history.history['unknown_sim']
    plt.figure(figsize=(10, 6))
    plt.plot(known_sim, 'b', label='Known sim')
    plt.plot(unknown_sim, 'r', label='Unknown sim')
    plt.xlabel('Epochs')
    plt.ylabel('Train set sim mean')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'Train set sim mean.png'))
    plt.close()


    loss = history.history['loss']
    plt.figure(figsize=(10, 6))
    plt.plot(loss, 'b', label='loss')
    plt.xlabel('Epochs')
    plt.ylabel('loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'loss.png'))
    plt.close()

    ce_loss = history.history['ce_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(ce_loss, 'b', label='ce_loss')
    plt.xlabel('Epochs')
    plt.ylabel('ce_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'ce_loss.png'))
    plt.close()
    
    entropy_loss = history.history['entropy_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(entropy_loss, 'b', label='entropy_loss')
    plt.xlabel('Epochs')
    plt.ylabel('entropy_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'entropy_loss.png'))
    plt.close()
    
    feature_loss = history.history['feature_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(feature_loss, 'b', label='feature_loss')
    plt.xlabel('Epochs')
    plt.ylabel('feature_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'feature_loss.png'))
    plt.close()
    
    ortho_loss = history.history['ortho_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(ortho_loss, 'b', label='ortho_loss')
    plt.xlabel('Epochs')
    plt.ylabel('ortho_loss')
    plt.legend()
    plt.savefig(os.path.join(save_dir, 'ortho_loss.png'))
    plt.close()
    



    save_history_to_csv(history.history['accuracy'], 'Class_Accuracy.csv',save_dir)
    save_history_to_csv(history.history['loss'], 'loss.csv',save_dir)
    save_history_to_csv(history.history['known_sim'], 'known_sim.csv',save_dir)
    save_history_to_csv(history.history['unknown_sim'], 'unknown_sim.csv',save_dir)
    
    save_history_to_csv(history.history['ce_loss'], 'ce_loss.csv',save_dir)
    save_history_to_csv(history.history['entropy_loss'], 'entropy_loss.csv',save_dir)
    save_history_to_csv(history.history['feature_loss'], 'feature_loss.csv',save_dir)
    save_history_to_csv(history.history['ortho_loss'], 'ortho_loss.csv',save_dir)

    


In [ ]:

# ======== Retrain the final model and evaluate on the test set ===================
test_dir = os.path.join(base_path, "test")
os.makedirs(test_dir, exist_ok=True)

tf.keras.backend.clear_session()
final_model = ObjectosphereRecognizer(
    num_known_classes = num_known_classes,
    seq_length = seq_length,
    channels = 2,
    margin = 0.6,
    alpha = 0.1,
    beta = 0.05,
    ortho_weight = ortho_weight_set
)
final_model.compile(optimizer=tf.keras.optimizers.Adam(0.001),metrics=['accuracy'])

print("===Start Training===\n")

history = final_model.fit(final_train_ds,epochs=nb_epoch,verbose=0)
save_result_test(test_dir,history)

test_ds = test_ds.unbatch().batch(batch_size).prefetch(tf.data.AUTOTUNE)
knwon_acc1_test, unknwon_acc1_test, auc_test = final_model.evaluate_oscr(test_ds,test_dir, threshold=0.4, use_similarity=True)
knwon_acc2_test, unknwon_acc2_test, auc_test2 = final_model.evaluate_oscr(test_ds,test_dir, threshold=0.4, use_similarity=False)
print(f"Independent test set OSCR AUC(Feature magnitude + probability): {auc_test:.4f}")
print(f"Independent test set OSCR AUC(Only probability): {knwon_acc1_test:.4f}")
print(f"Independent test set OSCR AUC(Only probability): {unknwon_acc1_test:.4f}")
summary_msg = (
    f"==== Test Set Result ====\n"
    f"Feature magnitude + probability: known_acc: {knwon_acc1_test:.4f}--unknown_acc: {unknwon_acc1_test:.4f}--auc_score: {auc_test:.4f}\n"
    f"Only probability: known_acc: {knwon_acc2_test:.4f}--unknown_acc: {unknwon_acc2_test:.4f}--auc_score: {auc_test2:.4f}"
)
summary_path = os.path.join(test_dir,"test_OSCR_summart.txt")
with open(summary_path,"w",encoding = "utf-8") as f:
    f.write(summary_msg)


features_list = []
probs_list = []
orth_loss_list = []
labels_list = []

for x_batch,y_batch in test_ds:
    probs_sub, features_sub, orth_loss_sub = model(x_batch,training = False)
    features_list.append(features_sub.numpy())
    probs_list.append(probs_sub.numpy())
    orth_loss_list.append(orth_loss_sub.numpy())
    labels_list.append(y_batch.numpy())


features_test = np.concatenate(features_list,axis = 0)
probs_test = np.concatenate(probs_list,axis = 0)
orth_loss_test = np.array(orth_loss_list)
y_test_fold_oh = np.concatenate(labels_list,axis = 0)


class_names = [f'Class {i}' for i in range(4)]
class_names.append('Unknown')
mat_file_path = os.path.join(test_dir,'features_and_labels.mat')
sio.savemat(mat_file_path,{
    'features':features_test,
    'labels':y_test_fold_oh,
    'class_names':class_names
})


visualize_feature_sphere(
    features_test,
    y_test_fold_oh,
    class_names=class_names,
    unknown_class_index=num_known_classes,
    save_path=f'{test_dir}/feature_sphere_visualization.png'
)